---
### Check if code is running in google colab or Kaggle

In [ ]:
import os

try:
  import google.colab
  IN_COLAB = True
  os.environ['colab_or_kaggle'] = 'True'
except:
  IN_COLAB = False # Currently only kaggle is supported in exception
  os.environ['colab_or_kaggle'] = 'False'

print(IN_COLAB)

---
## Download kaggle data in colab

In [ ]:
if IN_COLAB:
  # Info on how to get your api key (kaggle.json) here: https://github.com/Kaggle/kaggle-api#api-credentials
  %pip install kaggle
  api_token = {'username':'haldarankit','key':'e4a3e87eae6be921d669fdcbfee2d2bc'}

  !mkdir -p /root/.kaggle/

  import json
  with open('/root/.kaggle/kaggle.json', 'w') as file:
    json.dump(api_token, file)
  !chmod 600 /root/.kaggle/kaggle.json

In [ ]:
%%bash
competition_name='severstal-steel-defect-detection'
project_git_folder='all_projects'

# if in google colab
if [ $colab_or_kaggle == 'True' ]
then
  # Create folders similar to kaggle execution in colab
  mkdir -p '/content/kaggle'
  cd '/content/kaggle'

  data_path='/content/kaggle/input'
  code_path='/content/kaggle/working'
  echo $data_path
  mkdir -p ${data_path}
  mkdir -p ${code_path}

  # set kaggle data download path
  kaggle config set -n path -v $data_path

  # Download kaggle dataset in kaggle kernel format
  kaggle competitions download -c $competition_name -p $data_path
  unzip -q $data_path/$competition_name.zip -d $data_path/$competition_name/
  rm -rfv $data_path/$competition_name.zip

# if not in google colab (exception - currently using only kaggle)
# change if needed
else
  code_path='/kaggle/working'
fi


# download github repo
cd ${code_path}

MOVE_TO_BRANCH='competitions/kaggle-severstal'

# Delete in directory exists
if [ -d ${project_git_folder} ]; then
  rm -rf ${project_git_folder}
fi

# clone git repo
git clone https://github.com/ankithaldar/${project_git_folder}.git

# get into kaggle git repo
cd ${project_git_folder}

# more to another branch
git checkout -b $MOVE_TO_BRANCH refs/remotes/origin/$MOVE_TO_BRANCH
git pull

# check its contents
ls -al

In [ ]:
%pip install -r /content/kaggle/working/all_projects/requirements.txt
# %pip install -U 'torch_xla>=1.13

---
# EDA

### Installs

In [ ]:
%%bash


### Imports

In [ ]:
import pandas as pd
import numpy as np
import os
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
import cv2

### Constants

In [ ]:
DATA_ROOT = Path('/content/kaggle/input/severstal-steel-defect-detection')
CSV_PATH = DATA_ROOT/'train.csv'
IMAGES_DIR = DATA_ROOT/'train'

### Utility Functions

In [ ]:
def create_image_dataset():


In [ ]:
# ----------------------------
# RLE Decoding Utility
# ----------------------------
def rle_decode(mask_rle: str, shape: tuple = (256, 256)) -> np.ndarray:
  '''
  Decode RLE-encoded mask to binary mask.
  Args:
      mask_rle: Run-length as string formatted (start length)
      shape: (height, width) of array to return
  Returns:
      Binary mask as np.ndarray of shape (H, W)
  '''
  s = mask_rle.split()
  starts, lengths = [np.asarray(x, dtype=int) for x in (s[0::2], s[1::2])]
  starts -= 1
  ends = starts + lengths
  img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
  for lo, hi in zip(starts, ends):
      img[lo:hi] = 1
  return img.reshape(shape).T  # Transpose if needed based on your data